<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 09. LightGCN: 추천에는 전파만 남기기

## LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation

- **원 논문:** [LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation](https://arxiv.org/abs/2002.02126)
- **저자 / 발표:** Xiangnan He, Kuan Deng, Xiang Wang, Yan Li, Yongdong Zhang, Meng Wang · SIGIR (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** collaborative filtering, bipartite graph, BPR
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 비선형/feature transform 없이 user-item embedding을 전파하고 BPR로 학습합니다.

**축소하거나 생략한 부분:** Gowalla/Yelp2018/Amazon-Book 대신 6 user×9 item과 leave-one-out 평가를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1.1, Eq. (3)–(4) | 대칭 degree-normalized user↔item 전파 | 한 층 shape |
| §3.1.2, Eq. (7) | 0..K층 weighted sum | 층 평균 |
| Eq. (8) | user-item inner-product score | score matrix |
| §3.1.3, Eq. (9) | BPR loss | loss·Recall@3 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$e_u^{(k+1)}=\sum_{i\in\mathcal N_u}\frac{e_i^{(k)}}{\sqrt{|\mathcal N_u||\mathcal N_i|}},
\quad e_u=\sum_{k=0}^{K}\alpha_k e_u^{(k)}$$

- **기호 정의:** e는 user/item embedding, K는 전파 깊이, α는 층 결합 계수입니다.
- **수식의 역할:** feature transform과 활성화를 제거하고 협업 신호 전파 자체만 유지합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** Task P의 `preprocess_graph`, 논문 전용 클래스의
  `forward`, `compute_loss`, `training_step`이 §3.1.1, Eq. (3)–(4)을 구조화합니다.
- **입출력 shape:** user-item [U, I] → bipartite 전파 [U+I, U+I] → score [U, I]
- **평가:** BPR loss, Recall@3, 층별 embedding norm
- **원문 대비 한계:** Gowalla/Yelp2018/Amazon-Book 대신 6 user×9 item과 leave-one-out 평가를 사용합니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 논문 주장 확인 실험

- **원문 위치:** §3.1.3, Eq. (9)
- **이 셀의 책임:** BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
user_item = torch.tensor(raw["user_item"], dtype=torch.float32)
train_ui = user_item.clone()
heldout = []
for u in range(len(train_ui)):
    positive = torch.where(train_ui[u] > 0)[0]
    assert len(positive) >= 2
    item = int(positive[-1])
    heldout.append(item)
    train_ui[u, item] = 0
heldout = torch.tensor(heldout)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1.1, Eq. (3)–(4)
- **이 셀의 책임:** 대칭 degree-normalized user↔item 전파
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 한 층 shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def bipartite_normalized(ui: Tensor) -> Tensor:
    nu, ni = ui.shape
    a = torch.zeros(nu + ni, nu + ni, device=ui.device, dtype=ui.dtype)
    a[:nu, nu:] = ui
    a[nu:, :nu] = ui.T
    d = a.sum(1).clamp_min(1).pow(-0.5)
    return d[:, None] * a * d[None, :]


propagation = bipartite_normalized(train_ui)
assert torch.allclose(propagation, propagation.T)

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.1.2, Eq. (7)
- **이 셀의 책임:** 0..K층 weighted sum
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 층 평균. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def lightgcn_embeddings(initial: Tensor, prop: Tensor, layers: int):
    states = [initial]
    h = initial
    for _ in range(layers):
        h = prop @ h
        states.append(h)
    return torch.stack(states).mean(0), states


probe = torch.randn(len(propagation), 8)
final, states = lightgcn_embeddings(probe, propagation, 2)
assert len(states) == 3 and torch.allclose(
    final, (states[0] + states[1] + states[2]) / 3
)

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** Eq. (8) / §3.1.3, Eq. (9)
- **이 셀의 책임:** user-item inner-product score / BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** score matrix / loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
nu, ni = train_ui.shape
propagation_device, train_ui_device, heldout_device = ACCELERATOR.move(
    propagation, train_ui, heldout
)
initial = nn.Parameter(torch.randn(nu + ni, 10, device=DEVICE) * 0.1)
optimizer = torch.optim.Adam([initial], lr=0.05)
positives = [torch.where(train_ui[u] > 0)[0] for u in range(nu)]
negatives = [torch.where(user_item[u] == 0)[0] for u in range(nu)]
losses = []
for step in range(180):
    emb, _ = lightgcn_embeddings(initial, propagation_device, 2)
    ue, ie = emb[:nu], emb[nu:]
    pos = torch.tensor(
        [int(positives[u][step % len(positives[u])]) for u in range(nu)], device=DEVICE
    )
    neg = torch.tensor(
        [int(negatives[u][step % len(negatives[u])]) for u in range(nu)], device=DEVICE
    )
    pos_score = (ue * ie[pos]).sum(1)
    neg_score = (ue * ie[neg]).sum(1)
    loss = -F.logsigmoid(pos_score - neg_score).mean() + 1e-4 * initial.square().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
assert losses[-1] < losses[0]

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1.3, Eq. (9)
- **이 셀의 책임:** BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
with torch.no_grad():
    emb, states = lightgcn_embeddings(initial, propagation_device, 2)
    scores = emb[:nu] @ emb[nu:].T
    scores[train_ui_device.bool()] = -1e9
    top3 = scores.topk(3, dim=1).indices
    recall = float((top3 == heldout_device[:, None]).any(1).float().mean())
    norms = [float(s.norm(dim=1).mean()) for s in states]
assert 0 <= recall <= 1 and top3.shape == (nu, 3)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("BPR loss")
ax[1].plot(range(3), norms, marker="o")
ax[1].set_title("layer embedding norm")
plt.tight_layout()
plt.show()
print({"Recall@3": recall, "heldout": heldout.tolist(), "top3": top3.cpu().tolist()})

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 7 · 핵심 연산·모델

- **원문 위치:** §3.1.1, Eq. (3)–(4) / §3.1.2, Eq. (7) / Eq. (8) / §3.1.3, Eq. (9)
- **이 셀의 책임:** 대칭 degree-normalized user↔item 전파 / 0..K층 weighted sum / user-item inner-product score / BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 한 층 shape / 층 평균 / score matrix / loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    embedding_dim: int = 10
    layers: int = 2
    regularization: float = 1e-4
    recall_k: int = 3
    learning_rate: float = 0.03
    steps: int = 16


def preprocess_graph(interactions: Tensor) -> Tensor:
    user_count, item_count = interactions.shape
    graph = torch.zeros(user_count + item_count, user_count + item_count)
    graph[:user_count, user_count:] = interactions
    graph[user_count:, :user_count] = interactions.T
    degree = graph.sum(1).clamp_min(1.0).pow(-0.5)
    return degree[:, None] * graph * degree[None, :]


class LightgcnPortfolioModel(nn.Module):
    def __init__(self, user_count: int, item_count: int, config: PortfolioConfig):
        super().__init__()
        self.user_count = user_count
        self.config = config
        total = user_count + item_count
        self.embedding = nn.Parameter(torch.randn(total, config.embedding_dim) * 0.1)

    def forward(self, graph: Tensor) -> tuple[Tensor, Tensor]:
        states = [self.embedding]
        for _ in range(self.config.layers):
            states.append(graph @ states[-1])
        combined = torch.stack(states).mean(dim=0)
        return combined[: self.user_count], combined[self.user_count :]

    def compute_loss(self, graph: Tensor, positive: Tensor, negative: Tensor) -> Tensor:
        user_embedding, item_embedding = self(graph)
        user_ids = torch.arange(self.user_count, device=graph.device)
        positive_score = (user_embedding * item_embedding[positive]).sum(dim=1)
        negative_score = (user_embedding * item_embedding[negative]).sum(dim=1)
        ranking = -F.logsigmoid(positive_score - negative_score).mean()
        regularizer = self.embedding.square().mean()
        return ranking + self.config.regularization * regularizer

    def training_step(
        self,
        graph: Tensor,
        positive: Tensor,
        negative: Tensor,
    ) -> Tensor:
        return self.compute_loss(graph, positive, negative)


def sample_ranking_triplets(
    interactions: Tensor,
    heldout: Tensor,
    step: int,
) -> tuple[Tensor, Tensor]:
    positive = []
    negative = []
    for user_id, row in enumerate(interactions):
        observed = torch.where(row > 0)[0]
        missing = torch.where(row == 0)[0]
        missing = missing[missing != heldout[user_id]]
        assert len(observed) and len(missing)
        positive.append(observed[step % len(observed)])
        negative.append(missing[(step + user_id) % len(missing)])
    return torch.stack(positive), torch.stack(negative)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 8 · 목적함수·학습 update

- **원문 위치:** §3.1.3, Eq. (9)
- **이 셀의 책임:** BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: LightgcnPortfolioModel,
    graph: Tensor,
    interactions: Tensor,
    heldout: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    model.train()
    for step in range(model.config.steps):
        positive, negative = sample_ranking_triplets(interactions, heldout, step)
        positive, negative = ACCELERATOR.move(positive, negative)
        loss = model.training_step(graph, positive, negative)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 9 · 평가·완료 증거

- **원문 위치:** §3.1.3, Eq. (9)
- **이 셀의 책임:** BPR loss
- **Tensor shape 계약:** interaction [U,I] → E_u [U,D], E_i [I,D] → pair score [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·Recall@3. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: LightgcnPortfolioModel,
    graph: Tensor,
    interactions: Tensor,
    heldout: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        user_embedding, item_embedding = model(graph)
        scores = user_embedding @ item_embedding.T
        interactions_device, heldout_device = ACCELERATOR.move(interactions, heldout)
        scores = scores.masked_fill(interactions_device.bool(), -torch.inf)
        k = min(model.config.recall_k, scores.shape[1])
        recommendations = scores.topk(k, dim=1).indices
        recall = (recommendations == heldout_device[:, None]).any(dim=1).float().mean()
    return {f"recall_at_{k}": float(recall.cpu())}


portfolio_config = PortfolioConfig()
portfolio_graph = ACCELERATOR.move(preprocess_graph(train_ui))
portfolio_model = ACCELERATOR.move(
    LightgcnPortfolioModel(train_ui.shape[0], train_ui.shape[1], portfolio_config)
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_graph,
    train_ui,
    heldout,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_graph,
    train_ui,
    heldout,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["recall_at_3"] <= 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> Gowalla/Yelp2018/Amazon-Book 대신 6 user×9 item과 leave-one-out 평가를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.